# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yashcodes07/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [5]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Yashcodes07/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())

Working dir: /content/flyrank-ml-internship/flyrank-ml-internship


In [9]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)
df.columns.tolist()
print(df.columns.tolist())
df.head(3)

(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [11]:
lane_df = df[(df["impressions_prev_30d"] > 0) & (df["clicks_prev_30d"] >= 0)].copy()

lane_df["impression_decay_pct"] = (
    (lane_df["impressions_prev_30d"] - lane_df["impressions_last_30d"])
    / lane_df["impressions_prev_30d"]
) * 100

lane_df["click_decay_pct"] = (
    (lane_df["clicks_prev_30d"] - lane_df["clicks_last_30d"])
    / lane_df["clicks_prev_30d"].replace(0, pd.NA)
) * 100

# Composite: weight impression decay + click decay + how poor current position is
lane_df["opportunity_score"] = (
    lane_df["impression_decay_pct"].clip(lower=0) * 0.4
    + lane_df["click_decay_pct"].clip(lower=0).fillna(0) * 0.4
    + lane_df["avg_position"].clip(lower=0) * 0.2
)

# sanity check against the dataset's own trend_pct
lane_df[["content_id", "impression_decay_pct", "click_decay_pct", "avg_position",
         "trend_pct", "opportunity_score"]].sort_values("opportunity_score", ascending=False).head(10)


/tmp/ipykernel_1680/1909820151.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  + lane_df["click_decay_pct"].clip(lower=0).fillna(0) * 0.4


,content_id,impression_decay_pct,click_decay_pct,avg_position,trend_pct,opportunity_score
23814,content_c156a8ab04f5,96.551724,100.0,78.0,-96.6,94.220690
19348,content_cd918b354604,98.924731,100.0,69.7,-98.9,93.509892
14362,content_f68b1081eb71,98.431373,100.0,54.4,-98.4,90.252549
13100,content_611d55167bba,99.653979,100.0,48.7,-99.7,89.601592
795,content_2105b32488af,89.864865,100.0,58.6,-89.9,87.665946
20199,content_581f85ef23d3,84.225352,100.0,69.2,-84.2,87.530141
20348,content_df52e0cebadc,94.736842,100.0,46.1,-94.7,87.114737
8651,content_a50e3e16761d,88.235294,100.0,57.9,-88.2,86.874118
9648,content_9623e642ff3a,95.027829,100.0,44.1,-95.0,86.831132
26722,content_eb747ddae632,97.101449,100.0,35.9,-97.1,86.020580


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [13]:
top_by_score = set(lane_df.sort_values("opportunity_score", ascending=False).head(20)["content_id"])
top_by_trend = set(lane_df.sort_values("trend_pct").head(20)["content_id"])  # most negative trend_pct = worst decline

overlap = len(top_by_score & top_by_trend)
print(f"Overlap between opportunity_score top 20 and trend_pct top 20: {overlap}/20")


Overlap between opportunity_score top 20 and trend_pct top 20: 0/20


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [14]:
lane_df[
    ["content_id", "client_id", "avg_position", "impressions_last_30d", "impressions_prev_30d",
     "clicks_last_30d", "clicks_prev_30d", "days_since_last_update", "opportunity_score"]
].sort_values("opportunity_score", ascending=False).head(15)

,content_id,client_id,avg_position,impressions_last_30d,impressions_prev_30d,clicks_last_30d,clicks_prev_30d,days_since_last_update,opportunity_score
23814,content_c156a8ab04f5,client_f74efabef1,78.0,1,29,0,1,20,94.220690
19348,content_cd918b354604,client_9f14025af0,69.7,1,93,0,1,20,93.509892
14362,content_f68b1081eb71,client_f74efabef1,54.4,4,255,0,1,20,90.252549
13100,content_611d55167bba,client_a88a7902cb,48.7,1,289,0,1,28,89.601592
795,content_2105b32488af,client_9f14025af0,58.6,15,148,0,1,20,87.665946
20199,content_581f85ef23d3,client_6208ef0f77,69.2,168,1065,0,1,104,87.530141
20348,content_df52e0cebadc,client_f74efabef1,46.1,1,19,0,1,8,87.114737
8651,content_a50e3e16761d,client_e629fa6598,57.9,2,17,0,1,20,86.874118
9648,content_9623e642ff3a,client_6208ef0f77,44.1,134,2695,0,1,104,86.831132
26722,content_eb747ddae632,client_a88a7902cb,35.9,6,207,0,1,20,86.020580


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [15]:
naive_rule = lane_df[lane_df["days_since_last_update"] > 365]
top20_score = set(lane_df.sort_values("opportunity_score", ascending=False).head(20)["content_id"])
top20_naive = set(naive_rule.sort_values("opportunity_score", ascending=False).head(20)["content_id"])

print("Naive rule flags:", len(naive_rule), "pages")
print("Overlap between naive rule top 20 and opportunity_score top 20:", len(top20_score & top20_naive))

Naive rule flags: 3 pages
Overlap between naive rule top 20 and opportunity_score top 20: 0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.